# 임베딩 모델 벤치마크 (4종 비교)

KDIC RAG 파이프라인에 쓸 임베딩 모델을 정하기 위해 4개 모델을 성능(Recall/MRR)과 속도/자원(로딩·임베딩 시간, 메모리, 용량)으로 비교하고 순위를 매긴다.

- `Qwen/Qwen3-Embedding-8B`
- `nvidia/Nemotron-3-Embed-8B-BF16`
- `dragonkue/BGE-m3-ko`
- `BAAI/bge-m3`

**읽기 전용 원칙**: `src/` 아래 파이프라인 코드, `data/corpus.jsonl`, `data/chunks_all.jsonl`, `data/dense_cache/`는 이 노트북에서 절대 쓰지 않는다(전부 읽기만 함). 평가 로직은 `src/eval_retrieval.py`·`src/retrieval.py`에서 이 노트북 안으로 복사해 온 것이며 원본 파일은 건드리지 않는다. 결과는 전부 새 폴더 `data/embedding_test_jh/`에만 저장한다. git 명령은 실행하지 않는다.

**데이터**: `data/chunks_all.jsonl`(청크 494개, 58개 페이지)을 그대로 읽어서 청크 단위로 임베딩한다. 채점은 `src/retrieval.py`의 `PageRanked`와 같은 방식으로 청크 랭킹을 페이지 단위로 접어(Parent-Child 폴딩 — 같은 페이지의 여러 청크 중 최고 점수만 채택) `expected_sources`(페이지 id)와 비교한다. 일부 긴 청크는 모델의 최대 토큰 길이를 넘으면 자동으로 뒷부분이 절단된다.

**구조 (설계 원칙 — 모델 4개가 완전히 독립된 셀)**

아래 "모델 실행" 셀 4개는 각각 자기 안에서 에러를 전부 잡아서 절대 노트북 실행을 중단시키지 않는다. 한 모델(특히 8B급)이 GPU 메모리 부족으로 실패해도 그 사실만 출력하고 결과를 `status`에 기록한 뒤 다음 셀로 넘어간다. 즉:

- 4개 모델 셀은 순서·서로에 의존하지 않는다 — 아무 순서로나, 여러 번 다시 실행해도 안전하다.
- 8B 모델이 계속 OOM 나면 그 셀만 건너뛰고 나머지 3개만으로 먼저 순위를 봐도 된다.
- 런타임을 재시작해도 이전에 성공한 모델의 결과는 `data/embedding_test_jh/results_summary.json`에 남아있고, 마지막 순위 셀이 디스크에서 다시 읽어오므로 안 날아간다.
- 8B 모델(Qwen3-Embedding-8B, Nemotron-3-Embed-8B-BF16)은 GPU 메모리를 많이 써서, 코랩 런타임 하나에 8B 모델 1개씩만 올리고 나머지는 다음 세션에서 실행하는 걸 권장한다(둘 다 같은 런타임에서 연속 실행하면 메모리 파편화로 실패할 수 있음). 청크가 494개라 8B 모델은 임베딩에도 시간이 꽤 걸린다.

In [37]:
!pip install -q -U sentence-transformers transformers accelerate huggingface_hub pandas psutil

In [38]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # torch import 전에 설정해야 적용됨

import gc
import json
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
from sentence_transformers import SentenceTransformer

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"총 메모리: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB")
else:
    print("⚠️ GPU가 감지되지 않았습니다. 8B급 모델(Qwen3-Embedding-8B, Nemotron-3-Embed-8B-BF16)은 CPU에서 매우 느리거나 메모리 부족으로 실패할 수 있습니다.")
    print("   Colab 상단 메뉴: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 > GPU 로 설정하세요.")

GPU: NVIDIA A100-SXM4-40GB
총 메모리: 39.49 GiB


In [39]:
REPO_RAW_BASE = "https://raw.githubusercontent.com/likelion-8/kdic-crawler/main"

def load_text_readonly(relpath):
    """로컬(여러 후보 경로)에서 먼저 찾고, 없으면 GitHub raw에서 읽기 전용으로 가져온다.
    src/ 파이프라인이나 data/ 원본 파일은 여기서 절대 쓰지 않는다 — 읽기만 한다."""
    for base in (Path("."), Path("kdic-crawler"), Path("/content/kdic-crawler")):
        p = base / relpath
        if p.exists():
            return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(f"{REPO_RAW_BASE}/{relpath}") as resp:
        return resp.read().decode("utf-8")

chunk_lines = load_text_readonly("data/chunks_all.jsonl").strip().splitlines()
chunks = [json.loads(l) for l in chunk_lines]
chunk_ids = [r["chunk_id"] for r in chunks]
chunk_texts = [r["text"] for r in chunks]
chunk2page = {r["chunk_id"]: r["page_id"] for r in chunks}
assert len(chunk_ids) == len(set(chunk_ids)), "chunk_id 중복 발견"

testset_lines = load_text_readonly("data/testset/testset_all.jsonl").strip().splitlines()
testset = [json.loads(l) for l in testset_lines]
questions = [
    (r["question"], set(r["expected_sources"]), r.get("question_type"), r.get("must_include"))
    for r in testset if r.get("expected_sources")
]
n_pages = len(set(chunk2page.values()))
print(f"청크: {len(chunk_ids)}개 (페이지 {n_pages}개, data/chunks_all.jsonl 기준)")
print(f"평가 질문: {len(questions)}개 (expected_sources 있는 것만)")

청크: 494개 (페이지 58개, data/chunks_all.jsonl 기준)
평가 질문: 557개 (expected_sources 있는 것만)


In [ ]:
KS = (1, 3, 5, 10)


def encode_query_vec(model, question, api_style):
    """질의 인코딩만 모델별 비대칭 API로 분기.
    - nemotron : encode_query()  - qwen : prompt_name="query"  - bge : plain
    """
    if api_style == "query_document_methods":
        return model.encode_query([question], normalize_embeddings=True)[0]
    if api_style == "prompt_name":
        return model.encode([question], prompt_name="query", normalize_embeddings=True)[0]
    return model.encode([question], normalize_embeddings=True)[0]


def evaluate(doc_embeddings, chunk_ids, chunk2page, model, questions, ks=KS, api_style="plain"):
    """src/eval_retrieval.py의 Recall@k/MRR + src/retrieval.py의 PageRanked(Parent-Child 접기)
    로직을 그대로 옮긴 것. 🔧 [핵심 수정] 질의 인코딩을 api_style에 따라 비대칭 처리."""
    recall_hits = {k: 0 for k in ks}
    rr_sum = 0.0
    for question, expected, *_ in questions:
        qvec = np.asarray(encode_query_vec(model, question, api_style))   # 🔧 수정
        scores = doc_embeddings @ qvec
        order = np.argsort(-scores)
        seen_pages = {}
        for i in order:
            pid = chunk2page[chunk_ids[i]]
            if pid not in seen_pages:  # 청크는 점수순 → 첫 등장이 그 페이지 최고점(Parent-Child 접기)
                seen_pages[pid] = scores[i]
        ranked_pages = sorted(seen_pages, key=lambda p: seen_pages[p], reverse=True)
        rank = next((i + 1 for i, pid in enumerate(ranked_pages) if pid in expected), None)
        if rank is not None:
            rr_sum += 1.0 / rank
            for k in ks:
                if rank <= k:
                    recall_hits[k] += 1
    n = len(questions)
    result = {f"Recall@{k}": round(recall_hits[k] / n, 4) for k in ks}
    result["MRR"] = round(rr_sum / n, 4)
    return result


def _selftest_evaluate():
    class _FakeModel:
        """질문 텍스트 → 미리 정해둔 벡터. 실제 모델 없이 랭킹+Parent-Child 접기 로직만 검증."""
        def encode(self, texts, normalize_embeddings=True):
            table = {
                "q_top1": np.array([1.0, 0.0, 0.0]),
                "q_top2": np.array([0.0, 1.0, 0.0]),
                "q_top3": np.array([0.0, 0.0, 1.0]),
            }
            return np.array([table[t] for t in texts])

    chunk_ids = ["A#0", "A#1", "B", "C"]
    chunk2page = {"A#0": "A", "A#1": "A", "B": "B", "C": "C"}  # A는 청크 2개짜리 페이지
    doc_embeddings = np.array([
        [0.9, 0.1, 0.01],   # A#0
        [0.6, 0.05, 0.01],  # A#1
        [0.1, 0.8, 0.5],    # B
        [0.0, 0.0, 0.7],    # C
    ])
    fake_questions = [
        ("q_top1", {"A"}, None, None),
        ("q_top2", {"A"}, None, None),
        ("q_top3", {"A"}, None, None),
    ]
    # api_style 생략 → 기본 "plain" 이라 _FakeModel.encode 그대로 사용됨
    r = evaluate(doc_embeddings, chunk_ids, chunk2page, _FakeModel(), fake_questions, ks=(1, 3))
    assert r["Recall@1"] == round(1 / 3, 4), r
    assert r["Recall@3"] == 1.0, r
    expected_mrr = round((1 / 1 + 1 / 2 + 1 / 3) / 3, 4)
    assert r["MRR"] == expected_mrr, r
    print("selftest ok:", r)


_selftest_evaluate()


In [ ]:
EMBED_TEST_DIR = Path("data/embedding_test_jh")
EMBED_TEST_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = EMBED_TEST_DIR / "results_summary.json"

random.seed(42)
query_sample = random.sample(questions, min(20, len(questions)))


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def get_disk_size_gb(repo_id):
    try:
        from huggingface_hub import scan_cache_dir
        cache = scan_cache_dir()
        for repo in cache.repos:
            if repo.repo_id == repo_id:
                return round(repo.size_on_disk / (1024**3), 2)
    except Exception as e:
        print(f"    (디스크 용량 조회 실패: {e})")
    return None


def save_result(result):
    if RESULTS_PATH.exists():
        existing = {r["model"]: r for r in json.loads(RESULTS_PATH.read_text(encoding="utf-8"))}
    else:
        existing = {}
    existing[result["model"]] = result
    RESULTS_PATH.write_text(
        json.dumps(list(existing.values()), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"  💾 저장: {RESULTS_PATH}")


def run_one_model(name, repo_id, trust_remote_code=False, model_kwargs=None,
                  batch_size=4, api_style="plain"):   # 🔧 [수정] api_style 추가
    print(f"\n{'=' * 60}\n[{name}] 시작 (api_style={api_style})\n{'=' * 60}")
    model_kwargs = model_kwargs or {}
    result = {"model": name, "repo_id": repo_id, "api_style": api_style, "status": "ok"}
    model = None
    doc_embeddings = None
    try:
        cleanup()
        proc = psutil.Process()
        mem_before = proc.memory_info().rss / (1024**2)

        t0 = time.time()
        model = SentenceTransformer(repo_id, trust_remote_code=trust_remote_code, model_kwargs=model_kwargs)
        result["load_sec"] = round(time.time() - t0, 2)
        result["dim"] = model.get_sentence_embedding_dimension()
        print(f"  로딩 완료: {result['load_sec']}초, 차원={result['dim']}")

        # 🔧 [핵심 수정] 문서 인코딩: nemotron만 encode_document, 나머지는 plain
        t0 = time.time()
        if api_style == "query_document_methods":
            doc_embeddings = model.encode_document(
                chunk_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=batch_size
            )
        else:  # qwen 문서 / bge 문서 = plain
            doc_embeddings = model.encode(
                chunk_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=batch_size
            )
        doc_embeddings = np.asarray(doc_embeddings)
        result["embed_corpus_sec"] = round(time.time() - t0, 2)
        print(f"  청크 임베딩 완료: {result['embed_corpus_sec']}초 ({len(chunk_texts)}개 청크)")

        np.save(EMBED_TEST_DIR / f"{name}_chunk_embeddings.npy", doc_embeddings)

        # 🔧 [수정] 질의 인코딩도 api_style 전달
        eval_result = evaluate(doc_embeddings, chunk_ids, chunk2page, model, questions, api_style=api_style)
        result.update(eval_result)
        print(f"  평가: {eval_result}")

        per_query_ms = []
        for q, *_ in query_sample:
            t0 = time.time()
            encode_query_vec(model, q, api_style)   # 🔧 [수정] 실제 질의 API로 속도 측정
            per_query_ms.append((time.time() - t0) * 1000)
        result["query_ms_avg"] = round(sum(per_query_ms) / len(per_query_ms), 1)
        print(f"  쿼리당 평균: {result['query_ms_avg']}ms ({len(query_sample)}개 샘플)")

        mem_after = proc.memory_info().rss / (1024**2)
        result["peak_ram_mb"] = round(mem_after - mem_before, 1)
        result["peak_gpu_mb"] = (
            round(torch.cuda.max_memory_allocated() / (1024**2), 1) if torch.cuda.is_available() else None
        )
        result["disk_gb"] = get_disk_size_gb(repo_id)
        print(f"  ✅ [{name}] 완료")

    except Exception as e:
        result["status"] = f"{type(e).__name__}: {e}"
        print(f"  ❌ [{name}] 실패: {result['status']}")
        if "out of memory" in str(e).lower() or "outofmemory" in type(e).__name__.lower():
            print("     → GPU 메모리 부족(OOM). 런타임 재시작 후 이 모델 셀 하나만 다시 실행해보세요.")

    finally:
        del model
        del doc_embeddings
        cleanup()
        save_result(result)

    return result


## 모델 4개 실행 (완전히 독립된 셀)

아래 4개 셀은 순서 상관없이 실행 가능하고, 서로 의존하지 않는다. 하나가 실패해도(특히 8B 모델의 GPU OOM) 그 셀 안에서 에러가 잡히고 다음 셀로 넘어간다.

- 먼저 작은 모델 2개(BGE-m3, BGE-m3-ko)부터 실행 권장 — 몇 분 안에 끝남.
- 8B 모델 2개(Qwen3-Embedding-8B, Nemotron-3-Embed-8B-BF16)는 되도록 한 세션에 하나씩만. 같은 런타임에서 둘 다 연속 실행하면 메모리 파편화로 실패할 수 있다.
- 재실행해도 안전 — 같은 모델 이름으로 결과를 덮어쓸 뿐, 누적 저장 방식이라 다른 모델 결과는 유지된다.

In [42]:
# 모델 1/4 — BGE-m3 (다국어 베이스라인)
_ = run_one_model("BGE-m3", "BAAI/bge-m3")


[BGE-m3] 시작


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  로딩 완료: 5.96초, 차원=1024


/tmp/ipykernel_719/476605527.py:58: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  result["dim"] = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/124 [00:00<?, ?it/s]

  청크 임베딩 완료: 7.14초 (494개 청크)
  평가: {'Recall@1': 0.5889, 'Recall@3': 0.8384, 'Recall@5': 0.9138, 'Recall@10': 0.9695, 'MRR': 0.726}
  쿼리당 평균: 22.2ms (20개 샘플)
  ✅ [BGE-m3] 완료
  💾 저장: data/embedding_test_jh/results_summary.json


In [43]:
# 모델 2/4 — BGE-m3-ko (한국어 파인튜닝)
_ = run_one_model("BGE-m3-ko", "dragonkue/BGE-m3-ko")


[BGE-m3-ko] 시작


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  로딩 완료: 5.43초, 차원=1024


/tmp/ipykernel_719/476605527.py:58: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  result["dim"] = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/124 [00:00<?, ?it/s]

  청크 임베딩 완료: 7.14초 (494개 청크)
  평가: {'Recall@1': 0.6409, 'Recall@3': 0.8833, 'Recall@5': 0.9515, 'Recall@10': 0.9767, 'MRR': 0.7717}
  쿼리당 평균: 22.3ms (20개 샘플)
  ✅ [BGE-m3-ko] 완료
  💾 저장: data/embedding_test_jh/results_summary.json


In [ ]:
# 모델 3/4 — Qwen3-Embedding-8B (질의만 prompt_name="query")
_ = run_one_model(
    "Qwen3-Embedding-8B",
    "Qwen/Qwen3-Embedding-8B",
    trust_remote_code=True,
    model_kwargs={"torch_dtype": torch.bfloat16},
    batch_size=1,
    api_style="prompt_name",   # 🔧 [핵심 수정]
)


In [ ]:
# 모델 4/4 — Nemotron-3-Embed-8B-BF16 (전용 encode_query/encode_document)
_ = run_one_model(
    "Nemotron-3-Embed-8B-BF16",
    "nvidia/Nemotron-3-Embed-8B-BF16",
    trust_remote_code=True,
    model_kwargs={"torch_dtype": torch.bfloat16},
    batch_size=1,
    api_style="query_document_methods",   # 🔧 [핵심 수정]
)


## 최종 비교 및 순위

디스크에 저장된 `results_summary.json`을 다시 읽어서 순위를 매긴다 — 위 4개 셀을 전부 실행하지 않았어도, 지금까지 성공한 모델만으로 순위를 볼 수 있다. 이 셀은 몇 번이든 다시 실행 가능하다.

In [46]:
if not RESULTS_PATH.exists():
    print("아직 결과가 없습니다. 위 4개 모델 셀 중 최소 1개를 먼저 실행하세요.")
else:
    all_results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    df = pd.DataFrame(all_results)
    ok = df[df["status"] == "ok"].copy()
    failed = df[df["status"] != "ok"]

    if ok.empty:
        print("성공한 모델이 없습니다. 아래 실패 내역을 확인하세요.")
    else:
        ok["성능순위"] = ok["Recall@5"].rank(ascending=False, method="min").astype(int)
        ok["속도점수_초"] = ok["embed_corpus_sec"] + ok["query_ms_avg"] / 1000
        ok["속도순위"] = ok["속도점수_초"].rank(ascending=True, method="min").astype(int)

        print("=" * 70)
        print("성능 순위 (Recall@5 기준, 동률이면 MRR로 판단)")
        print("=" * 70)
        for _, row in ok.sort_values("성능순위").iterrows():
            print(f"  {int(row['성능순위'])}위  {row['model']:<28} Recall@5={row['Recall@5']:.3f}  MRR={row['MRR']:.3f}")

        print()
        print("=" * 70)
        print("속도 순위 (코퍼스 임베딩 시간 + 쿼리당 시간, 낮을수록 빠름)")
        print("=" * 70)
        for _, row in ok.sort_values("속도순위").iterrows():
            print(f"  {int(row['속도순위'])}위  {row['model']:<28} 임베딩={row['embed_corpus_sec']}초  쿼리당={row['query_ms_avg']}ms")

    if not failed.empty:
        print()
        print("실패한 모델")
        for _, row in failed.iterrows():
            print(f"  - {row['model']}: {str(row['status'])[:120]}")

    print()
    print("=" * 70)
    print("상세 비교표")
    print("=" * 70)
    cols = ["model", "status", "Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR",
            "load_sec", "embed_corpus_sec", "query_ms_avg", "peak_ram_mb", "peak_gpu_mb", "dim", "disk_gb"]
    col_names_ko = {
        "model": "모델", "status": "상태",
        "load_sec": "로딩(초)", "embed_corpus_sec": "임베딩(초)", "query_ms_avg": "쿼리당(ms)",
        "peak_ram_mb": "최대RAM(MB)", "peak_gpu_mb": "최대GPU(MB)", "dim": "차원", "disk_gb": "용량(GB)",
    }
    table = df.reindex(columns=[c for c in cols if c in df.columns]).rename(columns=col_names_ko)
    print(table.to_string(index=False))

성능 순위 (Recall@5 기준, 동률이면 MRR로 판단)
  1위  BGE-m3-ko                    Recall@5=0.952  MRR=0.772
  2위  Qwen3-Embedding-8B           Recall@5=0.928  MRR=0.760
  3위  Nemotron-3-Embed-8B-BF16     Recall@5=0.917  MRR=0.750
  4위  BGE-m3                       Recall@5=0.914  MRR=0.726

속도 순위 (코퍼스 임베딩 시간 + 쿼리당 시간, 낮을수록 빠름)
  1위  BGE-m3                       임베딩=7.14초  쿼리당=22.2ms
  2위  BGE-m3-ko                    임베딩=7.14초  쿼리당=22.3ms
  3위  Nemotron-3-Embed-8B-BF16     임베딩=24.39초  쿼리당=49.1ms
  4위  Qwen3-Embedding-8B           임베딩=28.44초  쿼리당=59.4ms

상세 비교표
                      모델 상태  Recall@1  Recall@3  Recall@5  Recall@10    MRR  로딩(초)  임베딩(초)  쿼리당(ms)  최대RAM(MB)  최대GPU(MB)   차원  용량(GB)
      Qwen3-Embedding-8B ok    0.6355    0.8600    0.9282     0.9785 0.7599   8.08   28.44     59.4        6.2    16037.9 4096   14.11
Nemotron-3-Embed-8B-BF16 ok    0.6158    0.8743    0.9174     0.9785 0.7499   9.08   24.39     49.1       -3.8    16692.4 4096   14.83
               BGE-m3-ko ok    0.6409    

## 결론

미리 써두지 않고, 위에서 나온 결과를 그대로 계산해서 결론을 낸다. 성공한 모델이 1개뿐이거나 없으면 그에 맞게 경고만 출력한다.

In [47]:
if not RESULTS_PATH.exists():
    print("아직 결과가 없습니다.")
else:
    all_results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    df = pd.DataFrame(all_results)
    ok = df[df["status"] == "ok"].copy()

    if ok.empty:
        print("성공한 모델이 없어 결론을 낼 수 없습니다. 위 실패 내역을 참고해 재시도하세요.")
    else:
        ok["속도점수_초"] = ok["embed_corpus_sec"] + ok["query_ms_avg"] / 1000
        best_perf = ok.sort_values(["Recall@5", "MRR"], ascending=False).iloc[0]
        best_speed = ok.sort_values("속도점수_초", ascending=True).iloc[0]

        print(f"성능 1위: {best_perf['model']} (Recall@5={best_perf['Recall@5']:.3f}, MRR={best_perf['MRR']:.3f})")
        print(f"속도 1위: {best_speed['model']} (임베딩={best_speed['embed_corpus_sec']}초, 쿼리당={best_speed['query_ms_avg']}ms)")
        print()

        if len(ok) < 4:
            missing = {"BGE-m3", "BGE-m3-ko", "Qwen3-Embedding-8B", "Nemotron-3-Embed-8B-BF16"} - set(ok["model"])
            print(f"참고: 4개 중 {len(ok)}개만 결과가 있습니다 (미실행/실패: {', '.join(missing)}).")
            print()

        if best_perf["model"] == best_speed["model"]:
            print(f"결론: {best_perf['model']}이(가) 성능·속도 모두에서 1위 — 별도 트레이드오프 없이 채택 가능.")
        else:
            gap = best_perf["Recall@5"] - ok[ok["model"] == best_speed["model"]]["Recall@5"].iloc[0]
            print(f"결론: 성능은 {best_perf['model']}, 속도/자원은 {best_speed['model']}이(가) 앞선다.")
            print(f"  두 모델의 Recall@5 차이는 {gap:.3f}.")
            if gap >= 0.03:
                print(f"  → 차이가 꽤 커서({gap:.3f}) 서빙 비용을 감수하더라도 성능 우선이면 {best_perf['model']} 추천.")
                print(f"  → GPU/지연시간 제약이 크면 {best_speed['model']}이 현실적 대안.")
            else:
                print(f"  → 성능 차이가 작아({gap:.3f}) 속도·자원이 더 가벼운 {best_speed['model']} 추천(비용 대비 효율).")

성능 1위: BGE-m3-ko (Recall@5=0.952, MRR=0.772)
속도 1위: BGE-m3 (임베딩=7.14초, 쿼리당=22.2ms)

결론: 성능은 BGE-m3-ko, 속도/자원은 BGE-m3이(가) 앞선다.
  두 모델의 Recall@5 차이는 0.038.
  → 차이가 꽤 커서(0.038) 서빙 비용을 감수하더라도 성능 우선이면 BGE-m3-ko 추천.
  → GPU/지연시간 제약이 크면 BGE-m3이 현실적 대안.
